In [ ]:
# Resolve paths consistently from the repository root or notebooks/.
import os
import sys
from pathlib import Path

PROJECT_ROOT = next((candidate for candidate in (Path.cwd(), *Path.cwd().parents)
                     if (candidate / 'src').is_dir() and (candidate / 'data').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Run this notebook from within the cloned repository.')
os.chdir(PROJECT_ROOT / 'notebooks')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
(PROJECT_ROOT / 'figures' / 'sm').mkdir(parents=True, exist_ok=True)


# Modeling crime in USA


**"Science may be described as the art of systematic over-simplification — the art of discerning what we may with advantage omit."** (Karl Popper, The Open Universe : An Argument for Indeterminism (1992), p. 44)

### MAIN VERSION



Refer bayesian_regression_mod for urban scaling related supp stuff

In [ ]:
#sets the notebook width
from IPython.core.display import display, HTML


display(HTML("<style>.container { width:75% !important; }</style>"))
%load_ext autoreload
%autoreload 2
%matplotlib inline
import warnings

warnings.filterwarnings('ignore')

## Packages

In [ ]:
# Standard Libraries
import sys
import os
import glob
import math

In [ ]:
import palettable

In [ ]:
# conda install pandas scipy statsmodels scikit-learn

In [ ]:
# Third party libraries
import numpy as np
import pandas as pd
from scipy import stats
from scipy import optimize
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.metrics import r2_score
from scipy import optimize
import arviz as az
#import pystan

In [ ]:
import seaborn as sns

In [ ]:
import matplotlib.mlab as ml
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import ticker
import matplotlib.ticker as plticker

def stdfigsize(scale=1, nx=1, ny=1, ratio=1.3):
    """
    Returns a tuple to be used as figure size.
    -------
    returns (7*ratio*scale*nx, 7.*scale*ny)
    By default: ratio=1.3
    If ratio<0 them ratio = golden ratio
    """
    if ratio < 0:
        ratio = 1.61803398875
    return((7*ratio*scale*nx, 7*scale*ny))

def stdrcparams(usetex=False):
    """
    Set several mpl.rcParams and sns.set_style for my taste.
    ----
    usetex = True
    ----
    """
    sns.set_style("white")
    sns.set_style({"xtick.direction": "in",
                 "ytick.direction": "in"})
    rcparams = {'text.usetex': usetex,
              'font.family': 'sans-serif',
              #'font.sans-serif': ['Times'],
#               'text.latex.unicode': True,
#               'text.latex.preamble': [r"\usepackage[T1]{fontenc}",
#                                       r"\usepackage{lmodern}",
#                                       r"\usepackage{amsmath}",
#                                       r"\usepackage{mathptmx}"
#                                       ],
              'axes.labelsize': 30,
              'axes.titlesize': 30,
              'axes.spines.left':True,
              'axes.spines.bottom':True,
              'axes.spines.top': False,
              'axes.spines.right':False,
              'ytick.left': 'on',
              'xtick.bottom': 'on',
              'xtick.labelsize': '25',
              'ytick.labelsize': '25',
              'axes.linewidth': 1.8,
              'xtick.major.width': 1.8,
              'xtick.minor.width': 1.8,
              'xtick.major.size': 14,
              'xtick.minor.size': 7,
              'xtick.major.pad': 10,
              'xtick.minor.pad': 10,
              'ytick.major.width': 1.8,
              'ytick.minor.width': 1.8,
              'ytick.major.size': 14,
              'ytick.minor.size': 7,
              'ytick.major.pad': 10,
              'ytick.minor.pad': 10,
              'axes.labelpad': 15,
              'axes.titlepad': 15,
              "xtick.direction":
                "out",
              "ytick.direction": "out"
              }
    mpl.rcParams.update(rcparams)

mpl.rcParams['lines.linewidth'] = 5
mpl.rcParams['lines.color'] = '#3690c0'

In [ ]:
stdrcparams(usetex=False)
figsize=stdfigsize(ratio=-1)
xs,ys=figsize
figsize

In [ ]:
# conda install -c conda-forge watermark

## Local modules

In [ ]:
import pandas as pd
import numpy as np

ticksnamemap={'Intercept':'Intercept',
              'Blacks':'B\&AA pop.',
              'Minorities':'Other pop.',
              'Whites':'White pop.',
              'Poverty':'Pop. in poverty',
              'Murder':'Murders',
              'Population':'Population',
              'Education':'Pop. w/o JH diploma',
              'Unemployment':'Pop. unemployed',

             }

urban_metrics=['Population', 'Whites', 'Blacks', 'Minorities', 'Poverty', 'Education',
               'Unemployment', 'Murder']

def load_data():
    data = pd.read_csv('../data/clean_dataframes/full_merged_data_1960_2020.csv',
               index_col=0)
    data = data[data.Year <= 2020]
    data = data[data.Year >= 1960]

    return data

def replace_nan_by_zero(data):
    data=data[data['County'].isin(np.unique(data[((data.Year==1960) )].County.values))]
    data = data[['County','State','Year','Whites', 'Blacks', 'Minorities',
              'Poverty','Unemployment','Education', 'Murder','Population']]
    data = data.dropna()
    data=data.reset_index(drop=True)
    for metric in urban_metrics:
        # data[metric]= data[metric].replace(np.nan, 0)
        data[metric]=data[metric].astype(int)
    return data

def to_msa(data):
    crosswalk=pd.read_csv('../data/cbsatocountycrosswalk.csv')
    crosswalk['state']=crosswalk['state']
    crosswalk=crosswalk[['countyname','state','msaname']]
    crosswalk['name']=crosswalk['countyname'].str.title()+', '+crosswalk['state']
    crosswalk['msaname']=crosswalk['msaname'].str.split(',').str[0].str.title()+', '+crosswalk['state']
    crosswalk['countyname']=crosswalk['name'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8').str.strip()
    crosswalk['msaname']=crosswalk['msaname'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8').str.strip()
    crosswalk=crosswalk[['countyname','msaname']]
    msamap=dict()
    for i in range(0,len(crosswalk)):
        msamap[crosswalk.loc[i].countyname]=crosswalk.loc[i].msaname

    data_msa=data.copy()
    data_msa['County']=data_msa['County'].str.normalize('NFKD').str.encode('ascii',
                                                                             errors='ignore').str.decode('utf-8').str.strip()
    data_msa['MSA']=data_msa['County'].map(msamap)
    data_msa.dropna(inplace=True)
    data_msa.reset_index(drop=True,inplace=True)
    data_msa=data_msa.groupby(['MSA','State','Year']).sum().reset_index()
    data=data_msa

    states_names=pd.read_csv('../data/state_abbreviations.csv')
    states_names['full_name']=states_names['Name']+', '+states_names['Abbreviation']
    data=data[~data.MSA.isin(states_names['full_name'])]

    return data

def data_log_transform(data,dropna=True):
    df=data.copy()

    for metric in ['Whites', 'Blacks', 'Minorities','Poverty','Unemployment','Education', 'Murder','Population','Non-White']:
        df[metric]=(df[metric]).apply(np.log10).replace(-np.inf,0)
    df['Year']=data['Year']
    df['MSA']=data['MSA']
    df['State']=data['State']

    if dropna:
        df = df.dropna().reset_index(drop=True)
    df.to_csv('../data/log_df.csv')
    return df

# Cobb-Douglas-like model

In [ ]:
from src import pystan_cobb_douglas_mod as cb

## Mathematical formulas

The Cobb-Douglas model is:

$H_{it}= A_{it}  \prod_{k} Y_k^{b_{kit}}\eta_{it}$,

Using the log-transformed variables, $h \equiv \log H$,  $y_k \equiv \log Y_k$, we can write:

$h_{it}=a_{it}  +\sum_{k} b_{kit} y_{kit} + \epsilon_{it}$

- Pooled model

**Model 1: Pooled model**

$h_{it}=a + \sum_{k} b_{k} y_{kit} + \epsilon_{it}$

- Time-effects models

**Model 2: Time-varying intercept**

$h_{it}=a_{t} + \sum_{k} b_{k} y_{kit} + \epsilon_{it}$

**Model 3: Time-varying slopes**

$h_{it}=a + \sum_{k} b_{kt} y_{kit} + \epsilon_{it}$

**Model 4: Time-varying intercept and slope**

$h_{it}=a_{t} + \sum_{k} b_{kt} y_{kit} + \epsilon_{it}$

- Entity-effects models

**Model 5: Time-varying intercept**

$h_{it}=a_{j} + \sum_{k} b_{k} y_{kit} + \epsilon_{it}$

**Model 6: Time-varying slopes**

$h_{it}=a +\sum_{k} b_{kj} y_{kit} + \epsilon_{it}$

**Model 7: Time-varying intercept and slope**

$h_{it}=a_{j} + \sum_{k} b_{kj} y_{kit} + \epsilon_{it}$

- Entity- and time-effects models

**Model 8: Time- and entity-varying intercept**

$h_{it}=a_{jt} + \sum_{k} b_{k} y_{kit} + \epsilon_{it}$


**Model 9: Time- and entity-varying intercepts and time-varying slopes**

$h_{it}=a_{jt} + \sum_{k} b_{kt} y_{kit} + \epsilon_{it}$

**Model 10: Time- and entity-varying intercepts and slopes**

$h_{it}=a_{jt} + \sum_{k} b_{kjt} y_{kit} + \epsilon_{it}$

In [ ]:
model_map_name={'model_1':'Pooled',
                'model_2':'Time-varying intercepts',
                'model_3':'Time-varying slopes',
                'model_4':'Time-varying intercepts\nand slopes',
                'model_5':'Entity-varying intercepts',
                'model_6':'Entity-varying slopes',
                'model_7':'Entity-varying intercepts\nand slopes',
                'model_8':'Time- and entity-varying intercepts',
                'model_9':'Time- and entity-varying intercepts\nand time-varing slopes',
                'model_10':'Time- and entity-varying\nintercepts and slopes',
                'model_11':'Random walk priors',
                'model_12':'Lasso priors',
                'model_13':'Pooled TL',
                'model_14':'Time-varying intercepts TL',
                'model_15':'Time-varying slopes TL',
                'model_16':'Time-varying intercepts\nand slopes TL',
                'model_17':'Entity-varying intercepts TL',
                'model_18':'Entity-varying slopes TL',
                'model_19':'Entity-varying intercepts\nand slopes TL',
                'model_20':'Time- and entity-varying intercepts TL',
                'model_21':'Time- and entity-varying intercepts\nand time-varing slopes TL',
                'model_22':'Time- and entity-varying\nintercepts and slopes TL',
                'model_23':'Random walk priors TL',
                'model_24':'RW - w/o white pop.  TL',
                'model_25':'RW - w/o B\&AA pop.  TL',
                'model_26':'RW - w/o other pop. TL',
                'model_27':'RW - w/o pop. in poverty TL',
                'model_28':'RW - w/o  pop. unemployed TL',
                'model_29':'RW - w/o  pop. w/o JH diploma TL',
                'model_30':'RWTL - 1990, w/o education',
                'model_31':'RWTL - 1990, with education',
                'model_32':'RWTL - 1970, all',
                'model_33':'RWTL - 1970, w/o education',
                'model_34':'RWTL - 1970, only demographics',
                'model_35':'RWTL - 1960, all (non-white)',
                'model_36':'RWTL - 1960, w/o education (non-white)',
                'model_37':'RWTL - 1960, white and non-white pop'

#                 'model_38':'RWTL - 1960, no initial priors',
#                 'model_39':'AR - 1960',
#                 'model_40':'RWTL - 1990, no initial priors',
#                 'model_41':'AR - 1990',

               }

In [ ]:
path='../data/log_df.csv'#'../data/log_df_centered.csv'
results_path='../results/5000/results/'

data=cb.read_data(path)

urban_metrics=['Murder']

model_cobb_douglas=dict()
trace_cobb_douglas=dict()
for dependent_var in urban_metrics:
#     print(dependent_var)
    model_cobb_douglas[dependent_var]=dict()
    trace_cobb_douglas[dependent_var]=dict()
    for i in range(1,38):
        model_name="model_{}".format(i)
        model_path=results_path+'cobb_douglas/{}/{}'.format(model_name,dependent_var)
        print(model_path)

        model,trace=cb.cobb_douglas(data,
                                    dependent_var,
                                    model_name,
                                    model_path,
                                 loadtrace=True)
        model_cobb_douglas[dependent_var][model_name],trace_cobb_douglas[dependent_var][model_name]=model,trace

In [ ]:
trace_cobb_douglas['Murder']['model_11']['waic'],trace_cobb_douglas['Murder']['model_12']['waic']

In [ ]:
trace_cobb_douglas['Murder']['model_11']['data']['b'].shape

## Convergence: $\hat{R}$

In [ ]:
ranges = [i for i in range(1,38)]#+[i for i in range(40,42)]
len(ranges)

In [ ]:
from src.stats_functions import cdf

In [ ]:
f, axs = plt.subplots(nrows=19, ncols=2, figsize=stdfigsize(nx=2, ny=10, scale=1),sharex=False)
colors={'Murder':'#e31a1c'}
count=0
for i in ranges:
    for var in ['Murder']:
        ax=axs.flatten()[count]
        df=trace_cobb_douglas[var]['model_{}'.format(i)]['summary'].reset_index()
        df=df[~df['index'].str.contains('yhat')]
        df=df[~df['index'].str.contains('log_lik')]

        bins, densities = cdf(df['Rhat'].values,nbins=40)
        ax.plot(bins, densities,color=colors[var])

        ax.axvline(1.1,color='k',ls='-')
        ax.text(1.1, 0.5, r"{:.2f}%".format(sum(df['Rhat'].values>1.1)/len(df['Rhat'].values)*100),
            color='k', fontsize=20,
            horizontalalignment="center", verticalalignment="center",
            bbox=dict(boxstyle="round", fc="white", ec='k', pad=0.2))
        ax.set_title(model_map_name['model_{}'.format(i)])
        ax.set_xlabel(r'$\hat{R}$')
        ax.set_ylabel(r'$S(\hat{R})$')
        count=count+1

for ax in axs.flatten()[-1:]:
    f.delaxes(ax)
f.tight_layout(h_pad=2,w_pad=3)


### Markov Chains check

In [ ]:
data_info=trace_cobb_douglas['Murder']['model_{}'.format(10)]['data']
a=np.round(data_info['sigma_b'].mean(axis=0),5)
# data_info

In [ ]:
data_info=trace_cobb_douglas['Murder']['model_{}'.format(10)]['data']
b=np.round(data_info['sigma_b'].mean(axis=0),5)
# data_info

In [ ]:
np.round(data_info['sigma_b'].mean(axis=0),5)

In [ ]:
np.round(data_info['mu_b'].mean(axis=0),5)

In [ ]:
data_info.keys()

In [ ]:
df=trace_cobb_douglas['Murder']['model_{}'.format(10)]['summary']
df[df['Rhat']>1.1]

In [ ]:
# f, axs = plt.subplots(nrows=1, ncols=2, figsize=stdfigsize(nx=3, ny=1, scale=0.75),sharex=False)
# param='mu'
# count=0

# ax = axs.flatten()[count]
# sns.distplot(data_info[param][:1000],ax=ax,color='#1f78b4')
# sns.distplot(data_info[param][1000:],ax=ax,color='#33a02c')
# ax.set_ylabel(r'Density')
# ax.set_xlabel(r'Parameter')
# count=count+1
# ax = axs.flatten()[count]
# ax.plot(data_info[param][:1000],color='#1f78b4',alpha=0.5)
# ax.plot(data_info[param][1000:],color='#33a02c',alpha=0.5)
# ax.set_ylabel(r'Parameter')
# ax.set_xlabel('MCMC step')
# count=count+1
# f.tight_layout()

In [ ]:
data_info['sigma_a']

In [ ]:
tslen=1000
f, axs = plt.subplots(nrows=7, ncols=2, figsize=stdfigsize(nx=3, ny=6, scale=0.75),sharex=False)
param='sigma_a'
count=0

ax = axs.flatten()[count]
sns.distplot(data_info[param][:tslen],ax=ax,color='#1f78b4')
sns.distplot(data_info[param][tslen:],ax=ax,color='#33a02c')
ax.set_ylabel(r'Density')
ax.set_xlabel(r'Parameter')
count=count+1
ax = axs.flatten()[count]
ax.plot(data_info[param][:tslen],color='#1f78b4')
ax.plot(data_info[param][tslen:],color='#33a02c',alpha=1)
ax.set_ylabel(r'Parameter')
ax.set_xlabel('MCMC step')
count=count+1


param='sigma_b'
for k in range(0,6):
    ax = axs.flatten()[count]

    sns.distplot(data_info[param][:,k][:tslen],ax=ax,color='#1f78b4')
    sns.distplot(data_info[param][:,k][tslen:],ax=ax,color='#33a02c')
    ax.set_ylabel(r'Density')
    ax.set_xlabel(r'Parameter')
    count=count+1
    ax = axs.flatten()[count]
    ax.plot(data_info[param][:,k][:tslen],color='#1f78b4')
    ax.plot(data_info[param][:,k][tslen:],color='#33a02c',alpha=1)
    ax.set_ylabel(r'Parameter')
    ax.set_xlabel('MCMC step')
    count=count+1

f.tight_layout()


In [ ]:
titles=['Intercept',
'Whites',
'Blacks',
'Minorities',
'Poverty',
'Unemployment',
'Education']
tslen=1000
f, axs = plt.subplots(nrows=7, ncols=2, figsize=stdfigsize(nx=3, ny=6, scale=0.75),sharex=False)

param='mu_a'
count=0

ax = axs.flatten()[count]
sns.distplot(data_info[param][:tslen],ax=ax,color='#1f78b4')
sns.distplot(data_info[param][tslen:],ax=ax,color='#33a02c')
ax.set_ylabel(r'Density')
ax.set_xlabel(r'Parameter')
ax.set_title(titles[0])
count=count+1
ax = axs.flatten()[count]
ax.plot(data_info[param][:tslen],color='#1f78b4')
ax.plot(data_info[param][tslen:],color='#33a02c',alpha=1)
ax.set_ylabel(r'Parameter')
ax.set_xlabel('MCMC step')
count=count+1
ax.set_title(titles[0])


param='mu_b'
for k in range(0,6):
    ax = axs.flatten()[count]

    sns.distplot(data_info[param][:,k][:tslen],ax=ax,color='#1f78b4')
    sns.distplot(data_info[param][:,k][tslen:],ax=ax,color='#33a02c')
    ax.set_ylabel(r'Density')
    ax.set_xlabel(r'Parameter')
    ax.set_title(titles[k+1])

    count=count+1
    ax = axs.flatten()[count]
    ax.plot(data_info[param][:,k][:tslen],color='#1f78b4')
    ax.plot(data_info[param][:,k][tslen:],color='#33a02c',alpha=1)
    ax.set_ylabel(r'{}'.format(param))
    ax.set_xlabel('MCMC step')
    ax.set_title(titles[k+1])

    count=count+1

f.tight_layout()


In [ ]:
data_info['b'].shape

In [ ]:

exog_var=[
 'Whites',
 'Blacks',
 'Minorities',
 'Poverty',
 'Unemployment',
 'Education']

entity=2
year=23
tslen=1000
f = plt.figure(figsize=stdfigsize(nx=3, ny=6, scale=0.75))

widths = [4,3]
heights = np.ones(7)
spec = f.add_gridspec(nrows=7,ncols=2,width_ratios=widths,height_ratios=heights)


ax = f.add_subplot(spec[0])

ax.plot(data_info['a'][:,entity][:,year][:tslen],color='#1f78b4',lw=2)
ax.plot(data_info['a'][:,entity][:,year][tslen:],color='#33a02c',alpha=0.5,ls='--',lw=2)
ax.set_title('Intercept')
ax.set_xlabel('MCMC step')
ax.set_ylabel(r'Parameter')


ax = f.add_subplot(spec[1])
sns.distplot(data_info['a'][:,entity][:,year][:tslen],color='#1f78b4',ax=ax)
sns.distplot(data_info['a'][:,entity][:,year][tslen:],color='#33a02c',ax=ax)
ax.set_title('Intercept')
ax.set_xlabel('Parameter')
ax.set_ylabel(r'PDF')

count=2
for j in range(0,6):
    ax = f.add_subplot(spec[count])
    ax.plot(data_info['b'][:,:,:,j][:,:,year][:,entity][:tslen],color='#1f78b4',lw=2)
    ax.plot(data_info['b'][:,:,:,j][:,:,year][:,entity][tslen:],color='#33a02c',alpha=0.5,ls='--',lw=2)
    ax.set_title(ticksnamemap[exog_var[j]])
    ax.set_xlabel('MCMC step')
    ax.set_ylabel(r'Parameter')
    count+=1

    ax = f.add_subplot(spec[count])
    sns.distplot(data_info['b'][:,:,:,j][:,:,year][:,entity][:tslen],color='#1f78b4',ax=ax)
    sns.distplot(data_info['b'][:,:,:,j][:,:,year][:,entity][tslen:],color='#33a02c',ax=ax)
    ax.set_title(ticksnamemap[exog_var[j]])
    ax.set_xlabel('Parameter')
    ax.set_ylabel(r'PDF')
    count+=1

# for ax in axs.flatten()[-1:]:
#     f.delaxes(ax)

f.tight_layout(h_pad=3)



## Load summaries

In [ ]:
cobb_douglas_summary=dict()
for dependent_var in ['Murder']:
    cobb_douglas_summary[dependent_var]=dict()
    for i in ranges:
        model_name="model_{}".format(i)
        print(model_name,dependent_var)
        cobb_douglas_summary[dependent_var][model_name]=trace_cobb_douglas[dependent_var][model_name]['summary']

## WAIC: Model comparison

In [ ]:
waic_results=dict()
for dependent_var in ['Murder']:
    print(dependent_var)
    waic_results[dependent_var]=dict()
    for i in ranges:
        model_name="model_{}".format(i)
        waic_results[dependent_var][model_name]=trace_cobb_douglas[dependent_var][model_name]['waic']

df_cobb_douglas_WAIC=dict()
for dependent_var in ['Murder']:
    results=[]
    for key in waic_results[dependent_var].keys():
        results.append(waic_results[dependent_var][key])
    df_cobb_douglas_WAIC[dependent_var]=pd.DataFrame(results,columns=['waic','se','p_waic'],
                                                      index=waic_results[dependent_var].keys())
    df_cobb_douglas_WAIC[dependent_var].index=df_cobb_douglas_WAIC[dependent_var].index.map(model_map_name)


In [ ]:
df_cobb_douglas_WAIC['Murder']

In [ ]:
df_subset1 = dict()
df_subset1['Murder'] =  df_cobb_douglas_WAIC['Murder'].iloc[0:12]
df_subset2 = dict()
df_subset2['Murder'] =  df_cobb_douglas_WAIC['Murder'].iloc[12:23]

In [ ]:
df_subset2['Murder']

In [ ]:
from src.plots import plot_model_selection

In [ ]:
df_cobb_douglas_WAIC['Murder'].sort_values(by='waic',ascending=False)

## WAIC: Variable selection

In [ ]:
df_cobb_douglas_WAIC['Murder'].reset_index()

In [ ]:
metric = 'Murder'

In [ ]:
#new addition

In [ ]:
import matplotlib as mpl
import matplotlib.colors as mcolors

mpl.rcParams.update({
    'text.usetex': False,
    'font.family': 'serif',
    'font.serif': ['DejaVu Serif'],
    'mathtext.fontset': 'dejavuserif',
    'font.size': 18,
    'axes.labelsize': 25,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
})

## Fig 3A

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams['text.usetex'] = False

color='#e31a1c'
f, ax = plt.subplots(nrows=1, ncols=1, figsize=stdfigsize(nx=1.2, ny=2.2, scale=0.75))

df=df_cobb_douglas_WAIC[metric].iloc[11:23].sort_values(by='waic',ascending=True)
waic=df['waic'].values
se=df['se'].values

x=np.arange(0,len(df))
ax.errorbar(waic,
            x,
            yerr=None,
            xerr=se,
            fmt='',
            ecolor=color,
            elinewidth=5,
            linewidth=0,
            capsize=10,
            barsabove=False, lolims=False, uplims=False, xlolims=False,
            xuplims=False, errorevery=1, capthick=3, data=None)
ax.plot(waic,x,'o',color=color,markersize=12)
ax.axvline(waic[-1],color='gray',ls='solid',lw=3,zorder=-1)
ax.set_yticks(np.arange(0,len(df)))
ax.set_yticklabels(labels=df.index.str.replace(' TL',''),rotation=0,ha='right',fontsize=20)
ax.set_title('Model selection',fontsize=30)
ax.set_xlabel('WAIC')
ax.annotate('A', xy=(-0.45, 1.0), xycoords="axes fraction",fontsize=40)
f.tight_layout(w_pad=0,h_pad=2)
f.savefig('../figures/fig3A.pdf',bbox_inches='tight')

In [ ]:
color='#e31a1c'
f, ax = plt.subplots(nrows=1, ncols=1, figsize=stdfigsize(nx=1.2, ny=2.2, scale=0.75))

df=df_cobb_douglas_WAIC[metric].iloc[22:29].sort_values(by='waic',ascending=True)
waic=df['waic'].values
se=df['se'].values

x=np.arange(0,len(df))
ax.errorbar(waic,
            x,
            yerr=None,
            xerr=se,
            fmt='',
            ecolor=color,
            elinewidth=5,
            linewidth=0,
            capsize=10,
            barsabove=False, lolims=False, uplims=False, xlolims=False,
            xuplims=False, errorevery=1, capthick=3, data=None)
ax.plot(waic,x,'o',color=color,markersize=12)
ax.axvline(waic[-1],color='gray',ls='solid',lw=3,zorder=-1)
ax.set_yticks(np.arange(0,len(df)))
ax.set_yticklabels(labels=df.index.str.replace(' TL',''),rotation=0,ha='right',fontsize=20)
ax.set_title('Feature selection',fontsize=30)
ax.set_xlabel('WAIC')
ax.annotate('B', xy=(-0.45, 1.0), xycoords="axes fraction",fontsize=40)
f.tight_layout(w_pad=0,h_pad=2)
f.savefig('../figures/fig3B.pdf',bbox_inches='tight')

# Time and entity varying parameters

In [ ]:
entities=np.unique(data['Regions'])
entity_dict = {i:entities[i] for i in range(len(entities))}
entity_dict

In [ ]:
model_map_name

In [ ]:
colors=palettable.colorbrewer.qualitative.Dark2_8.hex_colors
colors

## Fig 3CD

In [ ]:
dependent_var='Murder'
exog_var =['Blacks', 'Minorities','Poverty','Education']

f, axs = plt.subplots(nrows=2, ncols=4, figsize=stdfigsize(nx=3, ny=2.3, scale=0.75),sharex=True,sharey=False)


#Classical regression
formula='{} ~ Whites+Blacks+Minorities+Poverty+Unemployment+Education'.format(dependent_var)

results=[]
for year in range(1960,2021):
    model = smf.ols(formula,data=data,subset=data.Year==year)
    res = model.fit()
    results_as_html = res.summary().tables[1].as_html()
    results.append(pd.read_html(results_as_html, header=0, index_col=0)[0])

count=0
for j,var in enumerate(['Blacks', 'Minorities','Poverty','Education']):
    ax=axs.flatten()[j]
    vmean=[]
    vinf=[]
    vsup=[]
    for i in range(0,61):
        values=results[i].loc[var]
        vmean.append(values.loc['coef'])
        vinf.append(values.loc['[0.025'])
        vsup.append(values.loc['0.975]'])

    ax.plot(np.arange(1960,2021),vmean,color='gray',ls='dotted',lw=3,zorder=-1)
    ax.fill_between(np.arange(1960,2021),vinf,vsup,alpha=0.9,color='lightgray',zorder=-2)
#     ax.axhline(np.mean(values.loc['coef']),color='r',lw=2,ls='-')
    ax.axhline(0,color='k',lw=3,ls='dashed')
    count=count+1
    loc = plticker.MultipleLocator(base=.1) # this locator puts ticks at regular intervals
    ax.yaxis.set_major_locator(loc)
    if j in [2,3]:
        loc = plticker.MultipleLocator(base=.3) # this locator puts ticks at regular intervals
        ax.yaxis.set_major_locator(loc)
    loc = plticker.MultipleLocator(base=20) # this locator puts ticks at regular intervals
    ax.xaxis.set_major_locator(loc)
    ax.set_title(ticksnamemap[exog_var[j]],fontsize=30)


# ax.legend(labelspacing=0.15,frameon=True,loc='lower left',ncol=1,bbox_to_anchor=(-1, 0.5))
colors=palettable.colorbrewer.qualitative.Dark2_8.hex_colors
df=cobb_douglas_summary['Murder']['model_{}'.format(23)]


entities=np.unique(data['Regions'])
entity_dict = {i:entities[i] for i in range(len(entities))}
exog_var =['Blacks', 'Minorities','Poverty','Education']
exog_var = [1,2,3,5]
for j in [4,5,6,7]:
    e = exog_var[j-4]
    ax=axs.flatten()[j]
    regions_mean=[]
    for i in [2,1,0,3]:
        b=[]
        for t in range(1,62):
            param='{}[{},{},{}]'.format('b',i+1,t,e+1)
            b.append([df.loc[param]['mean'],df.loc[param]['2.5%'],df.loc[param]['97.5%']])
        b=np.array(b)
        ax.plot(np.arange(1960,2021),b[::,0],label=entity_dict[i],color=colors[i])
        ax.fill_between(np.arange(1960,2021),b[::,1],b[::,2],alpha=0.3,color=colors[i])
        regions_mean.append(df.loc[param]['mean'])
        ax.axhline(0,color='k',lw=3,ls='dashed')

ax.legend(labelspacing=0.15,frameon=True,loc='lower right',ncol=2,bbox_to_anchor=(1, -0.55))




axs.flatten()[0].annotate('C', xy=(-0.4, 1.01), xycoords="axes fraction",fontsize=40)
axs.flatten()[4].annotate('D', xy=(-0.4, 1.01), xycoords="axes fraction",fontsize=40)
axs.flatten()[4].set_xlabel('Year, $t$',fontsize=40)
axs.flatten()[5].set_xlabel('Year, $t$',fontsize=40)
axs.flatten()[6].set_xlabel('Year, $t$',fontsize=40)
axs.flatten()[7].set_xlabel('Year, $t$',fontsize=40)

axs.flatten()[0].set_ylabel(r'Exponent, $b_k$',fontsize=40)
axs.flatten()[4].set_ylabel(r'Exponent, $b_k$',fontsize=40)

f.tight_layout(w_pad=0,h_pad=2)


In [ ]:
dependent_var='Murder'
exog_var =['Blacks', 'Minorities','Poverty','Education']


"""
#Classical regression
formula='{} ~ Whites+Blacks+Minorities+Poverty+Unemployment+Education'.format(dependent_var)

results=[]
for year in range(1960,2021):
    model = smf.ols(formula,data=data,subset=data.Year==year)
    res = model.fit()
    results_as_html = res.summary().tables[1].as_html()
    results.append(pd.read_html(results_as_html, header=0, index_col=0)[0])

"""

f, axs = plt.subplots(nrows=3, ncols=4, figsize=stdfigsize(nx=3, ny=3, scale=0.75),sharex=True,sharey=False, constrained_layout = True)

count=0
#panel 1
for j,var in enumerate(['Blacks', 'Minorities','Poverty','Education']):
    ax=axs.flatten()[j]
    vmean=[]
    vinf=[]
    vsup=[]
    for i in range(0,61):
        values=results[i].loc[var]
        vmean.append(values.loc['coef'])
        vinf.append(values.loc['[0.025'])
        vsup.append(values.loc['0.975]'])

    ax.plot(np.arange(1960,2021),vmean,color='gray',ls='dotted',lw=3,zorder=-1)
    ax.fill_between(np.arange(1960,2021),vinf,vsup,alpha=0.9,color='lightgray',zorder=-2)
    ax.axhline(0,color='k',lw=3,ls='dashed')
    count=count+1
    loc = plticker.MultipleLocator(base=.1) # this locator puts ticks at regular intervals
    ax.yaxis.set_major_locator(loc)
    if j in [2,3]:
        loc = plticker.MultipleLocator(base=.3) # this locator puts ticks at regular intervals
        ax.yaxis.set_major_locator(loc)
    loc = plticker.MultipleLocator(base=20) # this locator puts ticks at regular intervals
    ax.xaxis.set_major_locator(loc)
    ax.set_title(ticksnamemap[exog_var[j]],fontsize=35, weight = 'bold')


colors=palettable.colorbrewer.qualitative.Dark2_8.hex_colors
df=cobb_douglas_summary['Murder']['model_{}'.format(23)]


entities=np.unique(data['Regions'])
entity_dict = {i:entities[i] for i in range(len(entities))}
exog_var =['Blacks', 'Minorities','Poverty','Education']
exog_var = [1,2,3,5]

#panel 2
for j in [4,5,6,7]:
    e = exog_var[j-4]
    ax1=axs.flatten()[j]
    regions_mean=[]
    for i in [2,0]:
        b=[]
        for t in range(1,62):
            param='{}[{},{},{}]'.format('b',i+1,t,e+1)
            b.append([df.loc[param]['mean'],df.loc[param]['2.5%'],df.loc[param]['97.5%']])
        b=np.array(b)
        ax1.plot(np.arange(1960,2021),b[::,0],label=entity_dict[i],color=colors[i])
        ax1.fill_between(np.arange(1960,2021),b[::,1],b[::,2],alpha=0.3,color=colors[i])
        regions_mean.append(df.loc[param]['mean'])
        ax1.axhline(0,color='k',lw=3,ls='dashed')


#panel 3
for j in [8,9,10,11]:
    e = exog_var[j-8]
    ax2=axs.flatten()[j]
    regions_mean=[]
    for i in [1,3]:
        b=[]
        for t in range(1,62):
            param='{}[{},{},{}]'.format('b',i+1,t,e+1)
            b.append([df.loc[param]['mean'],df.loc[param]['2.5%'],df.loc[param]['97.5%']])
        b=np.array(b)
        ax2.plot(np.arange(1960,2021),b[::,0],label=entity_dict[i],color=colors[i])
        ax2.fill_between(np.arange(1960,2021),b[::,1],b[::,2],alpha=0.3,color=colors[i])
        regions_mean.append(df.loc[param]['mean'])
        ax2.axhline(0,color='k',lw=3,ls='dashed')


#legend
axs.flatten()[4].legend(fontsize = 24, labelspacing=0.15,frameon=True,loc='upper left')
axs.flatten()[8].legend(fontsize = 24,labelspacing=0.15,frameon=True,loc='upper left')

#annotations
axs.flatten()[0].annotate('C', xy=(-0.55, 1.1), xycoords="axes fraction",fontsize=35)
axs.flatten()[4].annotate('D', xy=(-0.55, 1.1), xycoords="axes fraction",fontsize=35)

#xy labels
axs.flatten()[8].set_xlabel('Year, $t$',fontsize=35)
axs.flatten()[9].set_xlabel('Year, $t$',fontsize=35)
axs.flatten()[10].set_xlabel('Year, $t$',fontsize=35)
axs.flatten()[11].set_xlabel('Year, $t$',fontsize=35)

axs.flatten()[0].set_ylabel(r'Exponent, $b_k$',fontsize=35)
axs.flatten()[4].set_ylabel(r'Exponent, $b_k$',fontsize=35)
axs.flatten()[8].set_ylabel(r'Exponent, $b_k$',fontsize=35)
f.tight_layout()


In [ ]:
df=cobb_douglas_summary['Murder']['model_{}'.format(9)]
trace_cobb_douglas['Murder']['model_{}'.format(9)]['data']['b'].shape

In [ ]:
dependent_var='Murder'
exog_var =['Blacks', 'Minorities','Poverty','Education']


f, axs = plt.subplots(nrows=3, ncols=4, figsize=stdfigsize(nx=3, ny=3, scale=0.75),sharex=True,sharey=False, constrained_layout = True)

count=0
#panel 1
df=cobb_douglas_summary['Murder']['model_{}'.format(16)]


entities=np.unique(data['Regions'])
entity_dict = {i:entities[i] for i in range(len(entities))}
exog_var =['Whites', 'Blacks', 'Minorities','Poverty','Unemployment','Education']
j = 0
for i in [2,3,4,6]:
    ax=axs.flatten()[j]
    j+=1
    regions_mean=[]
    b=[]
    for t in range(1,62):
        param='{}[{},{}]'.format('b',t,i)
        b.append([df.loc[param]['mean'],df.loc[param]['2.5%'],df.loc[param]['97.5%']])
    b=np.array(b)
    ax.plot(np.arange(1960,2021),b[::,0],color='gray',ls = 'dotted', lw = 3, zorder=-1)
    ax.fill_between(np.arange(1960,2021),b[::,1],b[::,2],alpha=0.9,color='lightgray',zorder=-2)
    regions_mean.append(df.loc[param]['mean'])
    ax.axhline(0,color='k',lw=3,ls='dashed')
    loc = plticker.MultipleLocator(base=.3) # this locator puts ticks at regular intervals
    ax.yaxis.set_major_locator(loc)
    if j in [1,2,4]:
        loc = plticker.MultipleLocator(base=.1) # this locator puts ticks at regular intervals
        ax.yaxis.set_major_locator(loc)
    loc = plticker.MultipleLocator(base=20) # this locator puts ticks at regular intervals
    ax.xaxis.set_major_locator(loc)
    ax.set_title(ticksnamemap[exog_var[i-1]],fontsize=35)





colors=palettable.colorbrewer.qualitative.Dark2_8.hex_colors
df=cobb_douglas_summary['Murder']['model_{}'.format(23)]


entities=np.unique(data['Regions'])
entity_dict = {i:entities[i] for i in range(len(entities))}
exog_var =['Blacks', 'Minorities','Poverty','Education']
exog_var = [1,2,3,5]

#panel 2- CURRENT FIGURE HERE: WE USE THE BAYESIAN POOLED ESTIMATES
for j in [4,5,6,7]:
    e = exog_var[j-4]
    ax1=axs.flatten()[j]
    regions_mean=[]
    for i in [2,0]:
        b=[]
        for t in range(1,62):
            param='{}[{},{},{}]'.format('b',i+1,t,e+1)
            b.append([df.loc[param]['mean'],df.loc[param]['2.5%'],df.loc[param]['97.5%']])
        b=np.array(b)
        ax1.plot(np.arange(1960,2021),b[::,0],label=entity_dict[i],color=colors[i])
        ax1.fill_between(np.arange(1960,2021),b[::,1],b[::,2],alpha=0.3,color=colors[i])
        regions_mean.append(df.loc[param]['mean'])
        ax1.axhline(0,color='k',lw=3,ls='dashed')


#panel 3
for j in [8,9,10,11]:
    e = exog_var[j-8]
    ax2=axs.flatten()[j]
    regions_mean=[]
    for i in [1,3]:
        b=[]
        for t in range(1,62):
            param='{}[{},{},{}]'.format('b',i+1,t,e+1)
            b.append([df.loc[param]['mean'],df.loc[param]['2.5%'],df.loc[param]['97.5%']])
        b=np.array(b)
        ax2.plot(np.arange(1960,2021),b[::,0],label=entity_dict[i],color=colors[i])
        ax2.fill_between(np.arange(1960,2021),b[::,1],b[::,2],alpha=0.3,color=colors[i])
        regions_mean.append(df.loc[param]['mean'])
        ax2.axhline(0,color='k',lw=3,ls='dashed')


#legend
axs.flatten()[4].legend(fontsize = 24, labelspacing=0.15,frameon=True,loc='upper left')
axs.flatten()[8].legend(fontsize = 24,labelspacing=0.15,frameon=True,loc='upper left')

#annotations
axs.flatten()[0].annotate('C', xy=(-0.55, 1.1), xycoords="axes fraction",fontsize=35)
axs.flatten()[4].annotate('D', xy=(-0.55, 1.1), xycoords="axes fraction",fontsize=35)

#xy labels
axs.flatten()[8].set_xlabel('Year, $t$',fontsize=35)
axs.flatten()[9].set_xlabel('Year, $t$',fontsize=35)
axs.flatten()[10].set_xlabel('Year, $t$',fontsize=35)
axs.flatten()[11].set_xlabel('Year, $t$',fontsize=35)

axs.flatten()[0].set_ylabel(r'Exponent, $b_k$',fontsize=35)
axs.flatten()[4].set_ylabel(r'Exponent, $b_k$',fontsize=35)
axs.flatten()[8].set_ylabel(r'Exponent, $b_k$',fontsize=35)
f.tight_layout()
f.savefig('../figures/fig3CD.pdf',bbox_inches='tight')


# Observed vs. predicted


In [ ]:
from src.plots import plot_obs_pred

In [ ]:
path='../data/log_df.csv'
results_path='../results/5000/results/urban_scaling/'
data=cb.read_data(path)

urban_metrics=['Murder']

model_urban_scaling=dict()
trace_urban_scaling=dict()

for dependent_var in urban_metrics:
    print(dependent_var)
    model_urban_scaling[dependent_var]=dict()
    trace_urban_scaling[dependent_var]=dict()
    for i in range(1,23):
        model_name="model_{}".format(i)
        model_path=results_path+'{}/{}'.format(model_name,dependent_var)
        model,trace=cb.urban_scaling(df=data,
                                     dependent_var=dependent_var,
                                     model_key=model_name,
                                     save_path=model_path,
                                     loadtrace=True
                                    )
#         del trace['data']
        model_urban_scaling[dependent_var][model_name],trace_urban_scaling[dependent_var][model_name]=model,trace

## Fig4A

In [ ]:
us_waic = trace_urban_scaling['Murder']['model_22']['waic'][0]
cd_waic = trace_cobb_douglas['Murder']['model_23']['waic'][0]

In [ ]:
cd_waic = np.round(cd_waic, 0)
us_waic = np.round(us_waic,0)
cd_waic-us_waic

In [ ]:
cmap = mpl.cm.Blues(np.linspace(0,1,100))
cmap[0]=[1., 1., 1., 1.]
cmap = mpl.colors.ListedColormap(cmap[:,:-1])

In [ ]:
dependent_var='Murder'
model='model_22'
f, axs = plt.subplots(nrows=1, ncols=2, figsize=stdfigsize(nx=3, ny=1.5, scale=0.75),sharey=False,sharex=True)
ax=axs.flatten()[0]
data_info=trace_urban_scaling[dependent_var][model]['data']
ax=plot_obs_pred(data_info,data,dependent_var,f,ax,cmap)
ax.set_title('Urban scaling model')
ax.set_ylabel(r'Observed murders, $\log~H$')
ax.set_xlabel(r'Predicted murders, $\log~H$')
loc = plticker.MultipleLocator(base=1) # this locator puts ticks at regular intervals
ax.xaxis.set_major_locator(loc)
ax.yaxis.set_major_locator(loc)


ax=axs.flatten()[1]
model='model_23'
data_info=trace_cobb_douglas[dependent_var][model]['data']
ax=plot_obs_pred(data_info,data,dependent_var,f,ax,cmap)
ax.set_title('Cobb-Douglas model')
ax.set_ylabel(r'Observed murders, $\log~H$')
ax.set_xlabel(r'Predicted murders, $\log~H$')
loc = plticker.MultipleLocator(base=1) # this locator puts ticks at regular intervals
ax.xaxis.set_major_locator(loc)
ax.yaxis.set_major_locator(loc)


axs.flatten()[0].annotate('A', xy=(-0.35, 1.), xycoords="axes fraction",fontsize=40)
axs.flatten()[0].annotate(r'WAIC = {}'.format(int(us_waic)), xy=(0.05, 0.9), xycoords="axes fraction",fontsize=30)
axs.flatten()[1].annotate(r'WAIC = {}'.format(int(cd_waic)), xy=(0.05, 0.9), xycoords="axes fraction",fontsize=30)

#f.tight_layout(w_pad=3)
#f.savefig('../figures/4A.pdf',bbox_inches='tight')

## Fig 4B

In [ ]:
from src.plots import temporal_obs_pred

In [ ]:
cities=['Chicago, IL','New York-Newark, NY','Los Angeles-Long Beach, CA','Atlanta, GA']

f, axs = plt.subplots(nrows=2, ncols=2, figsize=stdfigsize(nx=3, ny=2.5, scale=0.75),sharex=True,sharey=True)
colors=['#1d91c0','#41ab5d','#3f007d','#fec44f']
for i in range(0,4):
    ax=axs.flatten()[i]
    dependent_var='Murder'
    data_info=trace_cobb_douglas[dependent_var]['model_23']['data']
    ax=temporal_obs_pred(data_info,data,dependent_var,cities[i],colors[i],ax)
    if i<3:
        ax.annotate(cities[i],xy=(0.05, 0.05), xycoords="axes fraction",fontsize=30,color=colors[i])
    else:
        ax.annotate(cities[i],xy=(0.05, 0.90), xycoords="axes fraction",fontsize=30,color=colors[i])

    if i in [2,3]:
        ax.set_xlabel('Year, $t$')
    if i in [0,2]:
        ax.set_ylabel('Murders, $\log~H(t)$')

#axs.flatten()[0].annotate('B', xy=(-0.20, 1.), xycoords="axes fraction",fontsize=40)

f.tight_layout()
f.savefig('../figures/fig4_no_poisson.pdf',bbox_inches='tight')

In [ ]:
#add poisson error bars

In [ ]:
# BLOCK 1 — PRECOMPUTE ONCE
# Cache all data needed for plotting.

poisson_plot_data = {}

dependent_var = 'Murder'
data_info = trace_cobb_douglas[dependent_var]['model_23']['data']
alpha = 0.05

cities = [
    'Chicago, IL',
    'New York-Newark, NY',
    'Los Angeles-Long Beach, CA',
    'Atlanta, GA'
]

colors = ['#1d91c0', '#41ab5d', '#3f007d', '#fec44f']

for city, color in zip(cities, colors):
    city_mask = data['MSA'].eq(city)

    years = data.loc[city_mask, 'Year'].to_numpy()
    log_H = data.loc[city_mask, dependent_var].to_numpy()

    # Murder is stored as log10(count).
    H = np.rint(10**log_H).astype(int)

    # Exact 95% Poisson (Garwood) confidence limits.
    ci_low = np.where(
        H > 0,
        0.5 * stats.chi2.ppf(alpha / 2, 2 * H),
        0.0
    )

    ci_high = 0.5 * stats.chi2.ppf(
        1 - alpha / 2,
        2 * (H + 1)
    )

    positive = H > 0

    years_plot = years[positive]
    y = log_H[positive]
    y_low = np.log10(ci_low[positive])
    y_high = np.log10(ci_high[positive])

    poisson_plot_data[city] = {
        'color': color,
        'years': years_plot,
        'y': y,
        'yerr': np.vstack((
            y - y_low,
            y_high - y
        ))
    }

In [ ]:
# BLOCK 2 — FAST STYLE PROTOTYPING
f, axs = plt.subplots(
    nrows=2,
    ncols=2,
    figsize=stdfigsize(nx=3, ny=2.5, scale=0.75),
    sharex=True,
    sharey=True
)

for i, city in enumerate(cities):
    ax = axs.flat[i]
    plot_data = poisson_plot_data[city]
    color = plot_data['color']

    # Strongly darkened city color
    rgb = np.asarray(mcolors.to_rgb(color))
    error_color = tuple(np.clip(rgb * 0.38, 0, 1))

    ax = temporal_obs_pred(
        data_info,
        data,
        dependent_var,
        city,
        color,
        ax
    )

    # Prediction line
    prediction_line = ax.lines[-2]
    prediction_line.set_linewidth(3.2)
    prediction_line.set_zorder(2)

    # White-centered observations
    observed_points = ax.lines[-1]
    observed_points.set_linestyle('none')
    observed_points.set_marker('o')
    observed_points.set_markerfacecolor(error_color)
    observed_points.set_markeredgecolor(error_color)
    observed_points.set_markeredgewidth(1.1)
    observed_points.set_markersize(6)
    observed_points.set_alpha(0.8)
    observed_points.set_zorder(4)

    # High-contrast Poisson intervals
    ax.errorbar(
        plot_data['years'],
        plot_data['y'],
        yerr=plot_data['yerr'],
        fmt='none',
        ecolor=error_color,
        elinewidth=1.0,
        capsize=2.0,
        capthick=1.0,
        alpha=1.0,
        zorder=3
    )

    annotation_y = 0.05 if i < 3 else 0.90

    ax.annotate(
        city,
        xy=(0.05, annotation_y),
        xycoords='axes fraction',
        fontsize=25,
        color=color,
        fontfamily='DejaVu Serif'
    )

    if i in [2, 3]:
        ax.set_xlabel(r'Year, $t$')

    if i in [0, 2]:
        ax.set_ylabel(r'Murders, $\log_{10} H(t)$')

    ax.tick_params(
        axis='both',
        which='major',
        labelsize=18,
        length=7,
        width=1.2
    )

f.tight_layout(w_pad=0.7, h_pad=0.5)
f.savefig('../figures/fig4op1.pdf', bbox_inches='tight')

In [ ]:
# BLOCK 2 — FAST STYLE PROTOTYPING
f, axs = plt.subplots(
    nrows=2,
    ncols=2,
    figsize=stdfigsize(nx=3, ny=2.5, scale=0.75),
    sharex=True,
    sharey=True
)

for i, city in enumerate(cities):
    ax = axs.flat[i]
    plot_data = poisson_plot_data[city]
    color = plot_data['color']

    # Strongly darkened city color
    rgb = np.asarray(mcolors.to_rgb(color))
    error_color = tuple(np.clip(rgb * 0.38, 0, 1))

    ax = temporal_obs_pred(
        data_info,
        data,
        dependent_var,
        city,
        color,
        ax
    )

    # Prediction line
    prediction_line = ax.lines[-2]
    prediction_line.set_linewidth(3.2)
    prediction_line.set_zorder(2)

    # White-centered observations
    observed_points = ax.lines[-1]
    observed_points.set_linestyle('none')
    observed_points.set_marker('o')
    observed_points.set_markerfacecolor('white')
    observed_points.set_markeredgecolor(error_color)
    observed_points.set_markeredgewidth(1.1)
    observed_points.set_markersize(4.8)
    observed_points.set_alpha(0.8)
    observed_points.set_zorder(4)

    # High-contrast Poisson intervals
    ax.errorbar(
        plot_data['years'],
        plot_data['y'],
        yerr=plot_data['yerr'],
        fmt='none',
        ecolor=error_color,
        elinewidth=1.0,
        capsize=2.0,
        capthick=1.0,
        alpha=1.0,
        zorder=3
    )

    annotation_y = 0.05 if i < 3 else 0.90

    ax.annotate(
        city,
        xy=(0.05, annotation_y),
        xycoords='axes fraction',
        fontsize=25,
        color=color,
        fontfamily='DejaVu Serif'
    )

    if i in [2, 3]:
        ax.set_xlabel(r'Year, $t$')

    if i in [0, 2]:
        ax.set_ylabel(r'Murders, $\log_{10} H(t)$')

    ax.tick_params(
        axis='both',
        which='major',
        labelsize=18,
        length=7,
        width=1.2
    )

f.tight_layout(w_pad=0.7, h_pad=0.5)
f.savefig('../figures/fig4.pdf', bbox_inches='tight')

In [ ]:
%watermark -n -u -v -iv -w